In [1]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum-bigdata', 'praktikum_bigdata']


In [2]:
!start-dfs.sh
!start-yarn.sh

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [fais]
Starting resourcemanager
Starting nodemanagers


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` (Sub-bab 10.2) sebelum mengerjakan latihan berikut.

In [3]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_latihan = "latihan10_stream"
shutil.rmtree(folder_latihan, ignore_errors=True)
os.makedirs(folder_latihan, exist_ok=True)

skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])
print("Siap untuk latihan. Folder:", folder_latihan)


26/11/13 02:17:29 WARN Utils: Your hostname, fais resolves to a loopback address: 127.0.1.1; using 10.112.153.43 instead (on interface wlp3s0)
26/11/13 02:17:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 02:17:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Siap untuk latihan. Folder: latihan10_stream


In [4]:
# Menyiapkan folder kosong yang akan berperan sebagai "sumber data streaming"
folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")

Folder simulasi streaming 'simulasi_stream_transaksi' siap (kosong).


In [5]:
# Mendefinisikan skema secara eksplisit — wajib untuk streaming source berbasis berkas
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema berhasil didefinisikan:")
print(skema_transaksi)

Skema berhasil didefinisikan:
StructType([StructField('trx_id', StringType(), True), StructField('kota', StringType(), True), StructField('kategori', StringType(), True), StructField('pendapatan', IntegerType(), True)])


**Soal 1.** Definisikan `readStream` dari folder `latihan10_stream` menggunakan `skema_latihan`, lalu buat agregasi `groupBy("kategori")` yang menghitung **rata-rata** `pendapatan` (gunakan `avg`, bukan `sum`). Jalankan query dengan `queryName="latihan_agg"`.

In [6]:
import os
from pyspark.sql.functions import avg

# Pastikan folder latihan10_stream ada
os.makedirs(folder_latihan, exist_ok=True)

# Hentikan query aktif dengan nama yang sama jika sel dijalankan ulang
for q in spark.streams.active:
    if q.name == "latihan_agg":
        q.stop()

# Membaca stream dari folder latihan10_stream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_latihan) \
    .option("header", True) \
    .load(folder_latihan)


# Agregasi rata-rata pendapatan berdasarkan kategori
df_agg = df_stream.groupBy("kategori").agg(
    avg("pendapatan").alias("rata_rata_pendapatan")
)

# Menjalankan streaming query ke memory sink
query = df_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("latihan_agg") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Getting offsets from FileStreamSource[file:/home/fais/praktikum-bigdata/latihan10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


**Soal 2.** Kirim (tulis) sebuah berkas CSV berisi minimal 3 baris data ke folder `latihan10_stream`, tunggu beberapa detik, lalu tampilkan isi tabel `latihan_agg` menggunakan `spark.sql()`.

In [7]:
import time
import os

# Menulis file CSV batch pertama (3 baris data)
data_batch1 = """trx_id,kategori,pendapatan
T001,Elektronik,1000000
T002,Elektronik,2000000
T003,Pakaian,500000
"""

with open(os.path.join(folder_latihan, "batch1.csv"), "w") as f:
    f.write(data_batch1)

# Tunggu beberapa detik agar micro-batch selesai diproses
time.sleep(5)

# Menampilkan isi tabel latihan_agg
spark.sql("SELECT * FROM latihan_agg").show()


+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|Elektronik|           1500000.0|
|   Pakaian|            500000.0|
+----------+--------------------+



**Soal 3.** Kirim **batch kedua** dengan kategori yang **sama** seperti salah satu kategori pada batch pertama, tunggu beberapa detik, lalu tampilkan ulang `latihan_agg`. Amati: apakah nilai rata-ratanya berubah sesuai perhitungan gabungan kedua batch?

In [8]:
# Menulis file CSV batch kedua dengan kategori yang sama (Elektronik)
data_batch2 = """trx_id,kategori,pendapatan
T004,Elektronik,3000000
T005,Makanan,200000
"""

with open(os.path.join(folder_latihan, "batch2.csv"), "w") as f:
    f.write(data_batch2)

# Tunggu beberapa detik agar batch kedua diproses
time.sleep(5)

# Menampilkan ulang isi tabel latihan_agg
spark.sql("SELECT * FROM latihan_agg").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|   Makanan|            200000.0|
|Elektronik|           2000000.0|
|   Pakaian|            500000.0|
+----------+--------------------+



**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa Spark Structured Streaming **mewajibkan** skema didefinisikan secara eksplisit (`StructType`), sementara pemrosesan batch pada Pertemuan 4 bisa memakai `inferSchema=True`? Tulis jawaban pada markdown cell di bawah ini.

**Spark Structured Streaming mewajibkan pendefinisian skema secara eksplisit karena sumber data streaming bersifat terus-menerus dan tak terbatas, sehingga Spark tidak memiliki seluruh kumpulan data di awal untuk dipindai guna menebak tipe data (inferSchema=True). Selain itu, skema eksplisit mencegah pembengkakan latensi yang tidak diperlukan sebelum aliran data diproses dan menjamin konsistensi struktur data agar query tidak gagal mendadak jika terjadi perubahan format pada data baru yang masuk.**


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas10_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan seluruh streaming query dihentikan (`query.stop()`) di akhir notebook sebelum dikumpulkan.
- Seluruh cell wajib sudah dijalankan (**Run All**) — perhatikan penggunaan `time.sleep()` yang cukup agar output streaming sempat terisi sebelum notebook mencoba menampilkannya.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Streaming Source | Skema & `readStream` didefinisikan dengan benar | 15% |
| B. Agregasi per Cabang | Kolom turunan & agregasi `groupBy` benar, query berjalan | 20% |
| C. Window Waktu | Agregasi window 2 menit terdefinisi & berjalan dengan benar | 20% |
| D. Pengujian Multi-Batch | Minimal 3 batch terkirim, hasil akumulasi terbukti benar | 25% |
| E. File Sink | Streaming ke Parquet berhasil, terverifikasi dapat dibaca kembali | 20% |


In [9]:
import os
import shutil
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as _sum, count, window

# Inisialisasi SparkSession
spark = (
    SparkSession.builder
    .appName("Tugas10_KopiNusantara_Streaming")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("SparkSession siap digunakan.")


SparkSession siap digunakan.


In [10]:
# Definisi skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Menyiapkan folder simulasi streaming (bersihkan sisa run sebelumnya jika ada)
folder_stream = "tugas10_stream"
for folder in [folder_stream, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(folder, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Mendefinisikan readStream dari folder tugas10_stream
df_stream = (
    spark.readStream
    .schema(skema_pesanan)
    .option("header", True)
    .csv(folder_stream)
)

print("Streaming source berhasil disiapkan pada folder:", folder_stream)


Streaming source berhasil disiapkan pada folder: tugas10_stream


In [11]:
# Menambahkan kolom total_harga
df_dengan_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi total pendapatan dan jumlah pesanan per cabang
df_agg_cabang = df_dengan_total.groupBy("nama_cabang").agg(
    _sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query pertama
query1 = (
    df_agg_cabang.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("agregasi_cabang")
    .start()
)

print("Query 1 (agregasi_cabang) berjalan...", query1.status)


Query 1 (agregasi_cabang) berjalan... {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [12]:
# Mendefinisikan ulang readStream kedua dari folder tugas10_stream
df_stream_2 = (
    spark.readStream
    .schema(skema_pesanan)
    .option("header", True)
    .csv(folder_stream)
    .withColumn("total_harga", col("qty") * col("harga_satuan"))
)

# Agregasi total_harga berdasarkan window waktu 2 menit
df_agg_waktu = df_stream_2.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    _sum("total_harga").alias("total_pendapatan_window")
)

# Menjalankan streaming query kedua
query2 = (
    df_agg_waktu.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("agregasi_waktu")
    .start()
)

print("Query 2 (agregasi_waktu) berjalan...", query2.status)


Query 2 (agregasi_waktu) berjalan... {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [13]:
# --- BATCH 1 ---
batch1 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P001,Jakarta,Kopi Susu Gula Aren,2,20000,2026-11-11 10:00:15
P002,Bandung,Americano,1,18000,2026-11-11 10:00:45
"""
with open(os.path.join(folder_stream, "batch1.csv"), "w") as f:
    f.write(batch1)

time.sleep(5)
print("=== Hasil Agregasi Cabang Setelah Batch 1 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()


# --- BATCH 2 ---
batch2 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P003,Jakarta,Caffe Latte,3,25000,2026-11-11 10:01:20
P004,Surabaya,Matcha Latte,2,24000,2026-11-11 10:01:50
"""
with open(os.path.join(folder_stream, "batch2.csv"), "w") as f:
    f.write(batch2)

time.sleep(5)
print("=== Hasil Agregasi Cabang Setelah Batch 2 (Jakarta terakumulasi) ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()


# --- BATCH 3 ---
batch3 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P005,Bandung,Kopi Susu Gula Aren,2,20000,2026-11-11 10:02:10
P006,Surabaya,Americano,1,18000,2026-11-11 10:02:40
"""
with open(os.path.join(folder_stream, "batch3.csv"), "w") as f:
    f.write(batch3)

time.sleep(5)
print("=== Hasil Agregasi Cabang Setelah Batch 3 (Semua cabang terakumulasi) ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

print("=== Hasil Agregasi Window 2 Menit (Query 2) ===")
spark.sql("SELECT window.start, window.end, total_pendapatan_window FROM agregasi_waktu ORDER BY window.start").show(truncate=False)


=== Hasil Agregasi Cabang Setelah Batch 1 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           18000|             1|
|    Jakarta|           40000|             1|
+-----------+----------------+--------------+

=== Hasil Agregasi Cabang Setelah Batch 2 (Jakarta terakumulasi) ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           18000|             1|
|    Jakarta|          115000|             2|
|   Surabaya|           48000|             1|
+-----------+----------------+--------------+

=== Hasil Agregasi Cabang Setelah Batch 3 (Semua cabang terakumulasi) ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           58000|             2|
|    Jakarta|          1150

In [14]:
# Menjalankan streaming query ketiga ke Parquet sink
query3 = (
    df_stream.writeStream
    .format("parquet")
    .option("path", "tugas10_output")
    .option("checkpointLocation", "tugas10_checkpoint")
    .outputMode("append")
    .start()
)

# Mengirim batch data baru (batch 4)
batch4 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P007,Jakarta,Caramel Macchiato,2,28000,2026-11-11 10:03:15
P008,Yogyakarta,Kopi Tubruk,3,15000,2026-11-11 10:03:45
"""
with open(os.path.join(folder_stream, "batch4.csv"), "w") as f:
    f.write(batch4)

# Tunggu beberapa detik agar file Parquet selesai ditulis
time.sleep(6)
print("Batch 4 berhasil diproses ke Parquet.")
# Membuktikan berkas Parquet tersimpan di folder lokal
!ls -la tugas10_output

# Membaca kembali berkas Parquet sebagai DataFrame batch biasa
df_hasil_parquet = spark.read.parquet("tugas10_output")
df_hasil_parquet.orderBy("pesanan_id").show()


Batch 4 berhasil diproses ke Parquet.
total 44
drwxr-xr-x  3 fais fais 4096 Nov 13 02:18 .
drwxrwxr-x 13 fais fais 4096 Nov 13 02:18 ..
-rw-r--r--  1 fais fais 1795 Nov 13 02:18 part-00000-edde6786-de6a-46be-8401-6d4f7368d62d-c000.snappy.parquet
-rw-r--r--  1 fais fais   24 Nov 13 02:18 .part-00000-edde6786-de6a-46be-8401-6d4f7368d62d-c000.snappy.parquet.crc
-rw-r--r--  1 fais fais 1789 Nov 13 02:18 part-00001-4c207261-be24-4d43-9abc-caec98134320-c000.snappy.parquet
-rw-r--r--  1 fais fais   24 Nov 13 02:18 .part-00001-4c207261-be24-4d43-9abc-caec98134320-c000.snappy.parquet.crc
-rw-r--r--  1 fais fais 1786 Nov 13 02:18 part-00002-4b624f35-d168-4df5-9899-645e805a2e0f-c000.snappy.parquet
-rw-r--r--  1 fais fais   24 Nov 13 02:18 .part-00002-4b624f35-d168-4df5-9899-645e805a2e0f-c000.snappy.parquet.crc
-rw-r--r--  1 fais fais 1770 Nov 13 02:18 part-00003-0e8bb5ff-9dba-4f87-aa91-005614ed8245-c000.snappy.parquet
-rw-r--r--  1 fais fais   24 Nov 13 02:18 .part-00003-0e8bb5ff-9dba-4f87-aa91-0

In [15]:
query1.stop()
query2.stop()
query3.stop()
spark.stop()
print("Semua streaming query dan SparkSession berhasil dihentikan.")


Semua streaming query dan SparkSession berhasil dihentikan.
